# Recommendation System - Degree-Normalised Combined Embeddings

This notebook combines semantic (BGE-M3) and structural (Node2Vec) embeddings to build a
recommendation system for WikiCS articles using cosine similarity.

### Key features:
1. **Power-Law Normalisation**: The Node2Vec component is divided by $\log(\text{degree}+1)^{\alpha}$
   where $\alpha$ is estimated via maximum-likelihood fitting of the degree distribution.
   This penalises high-degree hub nodes that would otherwise dominate recommendations.
2. **Popularity Bias Removal**: The degree normalisation ensures popular nodes are not
   over-represented purely due to their connectivity.
3. **Exact Title Matching**: All test queries use canonical Wikipedia article titles
   (e.g. `Machine_learning`, not `Machine Learning`).
4. **Fuzzy Fallback**: If an exact title is not found, `utils.resolve_title()` finds
   the shortest fuzzy match above a threshold.


In [ ]:
import sys
sys.path.insert(0, '../../utils')

import pickle, os, json
import numpy as np
import pandas as pd
import networkx as nx
from sklearn.metrics.pairwise import cosine_similarity

import warnings
warnings.filterwarnings('ignore')

from utils import (
    load_graph_data,
    fuzzy_search,
    resolve_title,
    load_embeddings,
    build_normalized_embeddings,
)

DATA_PATH     = "../../data/data-embeddings.json"
BGE_M3_PATH  = "../../cap-embeddings/BAAI_bge-m3/master_embeddings.parquet"
N2V_PATH     = "../../cap-embeddings/node2vec/master_embeddings.parquet"
CACHE_DIR    = "../../cache/recommendation-1"
os.makedirs(CACHE_DIR, exist_ok=True)

print("Libraries and utils loaded.")

## 1. Load Graph and Embeddings

In [ ]:
nodes, id_to_title, title_to_id, G = load_graph_data(DATA_PATH)
print(f"Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

df_bge, df_n2v = load_embeddings(BGE_M3_PATH, N2V_PATH)
print(f"BGE-M3  : {len(df_bge)} nodes, dim={len(df_bge['embedding'].iloc[0])}")
print(f"Node2Vec: {len(df_n2v)} nodes, dim={len(df_n2v['embedding'].iloc[0])}")

## 2. Power-Law Degree Distribution Fit

Estimate the power-law exponent $\alpha$ of the undirected degree distribution
using the `powerlaw` library.  The fitted $\alpha$ is then used to penalise
high-degree nodes in the Node2Vec component:

$$e_{\text{n2v}}^{*} = e_{\text{n2v}} / \log(d+1)^{\alpha}$$


In [ ]:
import powerlaw

degrees = [G.degree(n) for n in G.nodes()]

fit = powerlaw.Fit(np.array(degrees, dtype=float), discrete=True, verbose=False)
alpha = fit.power_law.alpha
xmin  = fit.power_law.xmin

print(f"Power-law fit results:")
print(f"  alpha (exponent) = {alpha:.4f}")
print(f"  xmin             = {xmin}")
print(f"  sigma (std err)  = {fit.power_law.sigma:.4f}")

ALPHA_FILE = os.path.join(CACHE_DIR, "alpha.pkl")
with open(ALPHA_FILE, "wb") as f:
    pickle.dump({"alpha": alpha, "xmin": xmin, "degrees": degrees}, f)
print(f"\nAlpha saved to {ALPHA_FILE}")

## 3. Build Degree-Normalised Combined Embeddings

Each node's final embedding is the concatenation of:
- **BGE-M3** (1024-dim): semantic content, unchanged
- **Node2Vec** (128-dim): divided by $\log(\text{degree}+1)^{\alpha}$ (popularity penalty)


In [ ]:
df_combined = build_normalized_embeddings(df_bge, df_n2v, G, alpha)
print(f"Combined embeddings: {len(df_combined)} nodes, dim={len(df_combined['embedding'].iloc[0])}")

EMB_FILE = os.path.join(CACHE_DIR, "combined_embeddings.pkl")
with open(EMB_FILE, "wb") as f:
    pickle.dump(df_combined, f)
print(f"Saved to {EMB_FILE}")

## 4. Precompute Similarity Matrix

In [ ]:
embedding_matrix = np.stack(df_combined["embedding"].values)
node_ids_arr = df_combined["id"].values
id_to_idx    = {nid: i for i, nid in enumerate(node_ids_arr)}
idx_to_id    = {i: nid for i, nid in enumerate(node_ids_arr)}
n_nodes      = len(node_ids_arr)

SIM_FILE = os.path.join(CACHE_DIR, "similarity_matrix.pkl")

if os.path.exists(SIM_FILE):
    print("  [cache] Loading similarity matrix...")
    with open(SIM_FILE, "rb") as f:
        sim_matrix = pickle.load(f)
    print(f"  [cache] Loaded: shape {sim_matrix.shape}")
else:
    print("  [comp]  Computing similarity matrix...")
    sim_matrix = cosine_similarity(embedding_matrix)
    with open(SIM_FILE, "wb") as f:
        pickle.dump(sim_matrix, f)
    print(f"  [save]  Saved.")

np.fill_diagonal(sim_matrix, 0.0)

existing_edges = set(G.edges())

def is_linked(src_id, tgt_id):
    return (src_id, tgt_id) in existing_edges

MAPPINGS_FILE = os.path.join(CACHE_DIR, "node_mappings.pkl")
with open(MAPPINGS_FILE, "wb") as f:
    pickle.dump({
        "node_ids_arr": node_ids_arr,
        "id_to_idx":    id_to_idx,
        "idx_to_id":    idx_to_id,
        "id_to_title":  id_to_title,
        "title_to_id":  title_to_id,
    }, f)

print(f"\nReady: {sim_matrix.shape}, {len(existing_edges)} edges")

## 5. Recommendation Functions

In [ ]:
def get_recommendations(query, top_k=10, verbose=True):
    """
    Get top-k recommendations for a query using degree-normalised cosine similarity.
    1. Try exact title match in title_to_id.
    2. Fall back to resolve_title() (fuzzy, shortest match wins).
    Returns list of dicts: {title, similarity, linked}.
    """
    matched = resolve_title(query, title_to_id)
    if matched is None:
        if verbose:
            print(f"No match found for '{query}'.")
        return None
    if matched != query and verbose:
        print(f"(resolved '{query}' -> '{matched}')")

    node_id = title_to_id[matched]
    idx     = id_to_idx[node_id]
    sims    = sim_matrix[idx]
    top_idx = np.argsort(sims)[::-1][:top_k]

    results = []
    for i in top_idx:
        rec_id = node_ids_arr[i]
        if rec_id == node_id:
            continue
        results.append({
            "title":      id_to_title[rec_id],
            "similarity": round(float(sims[i]), 6),
            "linked":     is_linked(node_id, rec_id),
        })

    return results[:top_k]


def display_recs(query, top_k=10):
    recs = get_recommendations(query, top_k=top_k)
    if recs is None:
        return
    print(f"\nTop-{top_k} for '{query}':")
    display(pd.DataFrame(recs))
    return recs

## 6. Exact-Match Test Queries

All queries below are exact Wikipedia article titles from the dataset.
They cover broad CS topics to test semantic understanding.

In [ ]:
# Canonical Wikipedia titles from the dataset
test_queries = [
    "Machine_learning",
    "Data_science",
    "Software_engineering",
    "Artificial_intelligence",
    "Computer_vision",
    "Natural_language_processing",
    "Deep_learning",
    "Algorithm",
    "Operating_system",
    "Computer_network",
    "Programming_language",
    "Database",
    "Cloud_computing",
    "Cybersecurity",
    "Quantum_computing",
]

print(f"Testing {len(test_queries)} queries...\n")
for q in test_queries:
    in_ds = q in title_to_id
    status = "EXACT" if in_ds else "FUZZY"
    recs = get_recommendations(q, top_k=5, verbose=False)
    if recs:
        tops = ", ".join([f"{r['title']} ({r['similarity']:.3f})" for r in recs])
        print(f"[{status}] {q}")
        print(f"       {tops}")
    else:
        print(f"[NO MATCH] {q}")
    print()

## 7. Detailed Recommendations

In [ ]:
display_recs("Machine_learning")
display_recs("Artificial_intelligence")
display_recs("Natural_language_processing")
display_recs("Deep_learning")

## 8. Batch Evaluation Table

In [ ]:
def get_top_10_table(titles):
    rows = []
    for title in titles:
        recs = get_recommendations(title, top_k=10, verbose=False)
        if recs is None:
            print(f"Warning: '{title}' not found.")
            continue
        row = {"Query": title}
        for i, res in enumerate(recs):
            flag = " [LINK]" if res["linked"] else ""
            row[f"Top {i+1}"] = f"{res['title']} ({res['similarity']:.4f}){flag}"
        rows.append(row)
    return pd.DataFrame(rows)

sample_titles = list(title_to_id.keys())[:20]
display(get_top_10_table(sample_titles))